In [42]:
import numpy as np 
import pandas as pd
import statsmodels.api as sm
from statsmodels.formula.api import ols
import os
import scipy.io as sio
import pingouin as pg
from statsmodels.stats.multitest import multipletests
from IPython.display import display

In [43]:
child_data_dir = "../children_data"
adult_data_dir = "../adult_data"

In [44]:
# get averages of any column across all participants in directory

def get_averages(data_directory, column_name, num_rows_to_process):
    files_in_dir = [f for f in os.listdir(data_directory) if f.endswith('.mat')]
    num_files = len(files_in_dir)

    S_matrix = np.full((num_rows_to_process, num_files), np.nan)

    for i, filename in enumerate(files_in_dir):
        file_path = os.path.join(data_directory, filename)
        mat_contents = sio.loadmat(file_path)
        data_column = mat_contents[column_name]

        squeezed_data = data_column.squeeze()
        processed_data = squeezed_data.flatten() if hasattr(squeezed_data, 'flatten') else np.array([squeezed_data])
        
        elements_to_extract = min(len(processed_data), num_rows_to_process)
        S_matrix[:elements_to_extract, i] = processed_data[:elements_to_extract]
            
    N = np.sum(~np.isnan(S_matrix), axis=1)
    std_dev = np.nanstd(S_matrix, axis=1, ddof=1)
    
    sem = np.zeros(num_rows_to_process)
    valid_n_for_sem = N > 1
    sem[valid_n_for_sem] = std_dev[valid_n_for_sem] / np.sqrt(N[valid_n_for_sem])

    S_matrix_for_mean = np.copy(S_matrix)
    S_matrix_for_mean[np.isnan(S_matrix_for_mean)] = 0 
    averages = np.mean(S_matrix_for_mean, axis=1)
            
    return averages, sem


def get_all_participant_values(data_dir, column, phase_idx):
    values = []
    for filename in os.listdir(data_dir):
        if filename.endswith('.mat'):
            mat = sio.loadmat(os.path.join(data_dir, filename))
            col = mat[column].squeeze()
            # Defensive: handle both 1D and 2D
            if col.ndim > 0 and len(col) > phase_idx:
                values.append(col[phase_idx])
    return np.array(values)

def get_group_data(column, n):
    child_mean, child_sem = get_averages(child_data_dir, column, n)
    adult_mean, adult_sem = get_averages(adult_data_dir, column, n)
    return child_mean[:3], child_sem[:3], adult_mean[:3], adult_sem[:3]

In [45]:
get_all_participant_values(child_data_dir, 'meanMT', 0)

array([0.8905    , 1.0164    , 1.0788    , 0.971     , 1.0929    ,
       0.9059    , 0.8487    , 1.1185    , 1.00777778, 0.8638    ,
       0.8114    ])

In [46]:
mt_df = pd.DataFrame({
    'group': ['children'] * 11 + ['adults'] * 13,
    'baseline_meanMT': np.append(get_all_participant_values(child_data_dir, 'meanMT', 0), get_all_participant_values(adult_data_dir, 'meanMT', 0)),
    'early_learning_meanMT': np.append(get_all_participant_values(child_data_dir, 'meanMT', 1), (get_all_participant_values(adult_data_dir, 'meanMT', 1))),
    'late_learning_meanMT': np.append(get_all_participant_values(child_data_dir, 'meanMT', 2), get_all_participant_values(adult_data_dir, 'meanMT', 2)),
})

df_mt_long = pd.melt(
    mt_df,
    id_vars=['group'],
    value_vars=['baseline_meanMT', 'early_learning_meanMT', 'late_learning_meanMT'],
    var_name='phase',
    value_name='meanMT'
)

df_mt_long['subject'] = np.tile(np.arange(len(mt_df)), 3)

aov_mt = pg.mixed_anova(
    dv='meanMT',
    within='phase',
    between='group',
    subject='subject',
    data=df_mt_long
)

# df_mt_long
print(aov_mt)

        Source        SS  DF1  DF2        MS          F         p-unc  \
0        group  0.021080    1   22  0.021080   1.052468  3.160878e-01   
1        phase  0.201493    2   44  0.100746  25.025407  5.520554e-08   
2  Interaction  0.018865    2   44  0.009433   2.343053  1.079066e-01   

   p-GG-corr       np2       eps sphericity  W-spher   p-spher  
0        NaN  0.045655       NaN        NaN      NaN       NaN  
1   0.000002  0.532168  0.764257      False  0.69154  0.017298  
2        NaN  0.096251       NaN        NaN      NaN       NaN  


In [47]:
rt_df = pd.DataFrame({
    'group': ['children'] * 11 + ['adults'] * 13,
    'baseline_meanRT': np.append(get_all_participant_values(child_data_dir, 'meanRT', 0), get_all_participant_values(adult_data_dir, 'meanRT', 0)),
    'early_learning_meanRT': np.append(get_all_participant_values(child_data_dir, 'meanRT', 1), get_all_participant_values(adult_data_dir, 'meanRT', 1)),
    'late_learning_meanRT': np.append(get_all_participant_values(child_data_dir, 'meanRT', 2), get_all_participant_values(adult_data_dir, 'meanRT', 2)),
})

df_rt_long = pd.melt(
    rt_df,
    id_vars=['group'],
    value_vars=['baseline_meanRT', 'early_learning_meanRT', 'late_learning_meanRT'],
    var_name='phase',
    value_name='meanRT'
)

df_rt_long['subject'] = np.tile(np.arange(len(rt_df)), 3)

aov_rt = pg.mixed_anova(
    dv='meanRT',
    within='phase',
    between='group',
    subject='subject',
    data=df_rt_long
)
print(aov_rt)

        Source        SS  DF1  DF2        MS         F     p-unc       np2  \
0        group  0.225317    1   22  0.225317  8.947897  0.006729  0.289128   
1        phase  0.001869    2   44  0.000934  0.758509  0.474390  0.033329   
2  Interaction  0.000086    2   44  0.000043  0.034878  0.965750  0.001583   

        eps  
0       NaN  
1  0.809824  
2       NaN  


In [48]:
import pingouin as pg

# Pairwise comparisons between blocks for meanMT within each group
pairwise_mt_within_group = pg.pairwise_tests(
    dv='meanMT', within='phase', between='group', subject='subject', data=df_mt_long, padjust='fdr_bh',
    parametric=True, effsize='hedges'
)
print("movement time pairwise (between blocks within each group)")
print(pairwise_mt_within_group)

# Pairwise comparisons between blocks for meanMT (ignoring group)
pairwise_mt_block = pg.pairwise_tests(
    dv='meanMT', within='phase', subject='subject', data=df_mt_long, padjust='fdr_bh',
    parametric=True, effsize='hedges'
)
print("movement time pairwise (between blocks, all subjects)")
print(pairwise_mt_block)

movement time pairwise (between blocks within each group)
        Contrast                  phase                      A  \
0          phase                      -        baseline_meanMT   
1          phase                      -        baseline_meanMT   
2          phase                      -  early_learning_meanMT   
3          group                      -                 adults   
4  phase * group        baseline_meanMT                 adults   
5  phase * group  early_learning_meanMT                 adults   
6  phase * group   late_learning_meanMT                 adults   

                       B Paired Parametric         T        dof alternative  \
0  early_learning_meanMT   True       True -6.164530  23.000000   two-sided   
1   late_learning_meanMT   True       True -4.098014  23.000000   two-sided   
2   late_learning_meanMT   True       True  2.749089  23.000000   two-sided   
3               children  False       True  1.029861  21.630605   two-sided   
4               ch

In [49]:

# Pairwise comparisons between blocks for meanRT (Python equivalent of R code)

# Pairwise comparisons between blocks for meanRT within each group
pairwise_rt_within_group = pg.pairwise_tests(
    dv='meanRT', within='phase', between='group', subject='subject', data=df_rt_long, padjust='fdr_bh',
    parametric=True, effsize='hedges'
)
print("response time pairwise (between blocks within each group)")
print(pairwise_rt_within_group)

# Pairwise comparisons between blocks for meanRT (ignoring group)
pairwise_rt_block = pg.pairwise_tests(
    dv='meanRT', within='phase', subject='subject', data=df_rt_long, padjust='fdr_bh',
    parametric=True, effsize='hedges'
)
print("response time pairwise (between blocks, all subjects)")
print(pairwise_rt_block)

response time pairwise (between blocks within each group)
        Contrast                  phase                      A  \
0          phase                      -        baseline_meanRT   
1          phase                      -        baseline_meanRT   
2          phase                      -  early_learning_meanRT   
3          group                      -                 adults   
4  phase * group        baseline_meanRT                 adults   
5  phase * group  early_learning_meanRT                 adults   
6  phase * group   late_learning_meanRT                 adults   

                       B Paired Parametric         T        dof alternative  \
0  early_learning_meanRT   True       True  1.362648  23.000000   two-sided   
1   late_learning_meanRT   True       True  0.772789  23.000000   two-sided   
2   late_learning_meanRT   True       True -0.291597  23.000000   two-sided   
3               children  False       True -2.829971  13.736075   two-sided   
4               ch

In [50]:
# For movement time (meanMT) pairwise comparisons (within each group)
pvals_mt = pairwise_mt_within_group['p-unc'].values
reject_mt, pvals_mt_fdr, _, _ = multipletests(pvals_mt, alpha=0.05, method='fdr_bh')
pairwise_mt_within_group['p-FDR'] = pvals_mt_fdr
print("Movement Time Pairwise Comparisons with FDR-corrected p-values:")
print(pairwise_mt_within_group)

# For response time (meanRT) pairwise comparisons (within each group)
pvals_rt = pairwise_rt_within_group['p-unc'].values
reject_rt, pvals_rt_fdr, _, _ = multipletests(pvals_rt, alpha=0.05, method='fdr_bh')
pairwise_rt_within_group['p-FDR'] = pvals_rt_fdr
print("\nResponse Time Pairwise Comparisons with FDR-corrected p-values:")
print(pairwise_rt_within_group)

Movement Time Pairwise Comparisons with FDR-corrected p-values:
        Contrast                  phase                      A  \
0          phase                      -        baseline_meanMT   
1          phase                      -        baseline_meanMT   
2          phase                      -  early_learning_meanMT   
3          group                      -                 adults   
4  phase * group        baseline_meanMT                 adults   
5  phase * group  early_learning_meanMT                 adults   
6  phase * group   late_learning_meanMT                 adults   

                       B Paired Parametric         T        dof alternative  \
0  early_learning_meanMT   True       True -6.164530  23.000000   two-sided   
1   late_learning_meanMT   True       True -4.098014  23.000000   two-sided   
2   late_learning_meanMT   True       True  2.749089  23.000000   two-sided   
3               children  False       True  1.029861  21.630605   two-sided   
4           

In [51]:
# Bonferroni correction for movement time and response time
_, pvals_mt_bonf, _, _ = multipletests(pairwise_mt_within_group['p-unc'], alpha=0.05, method='bonferroni')
_, pvals_rt_bonf, _, _ = multipletests(pairwise_rt_within_group['p-unc'], alpha=0.05, method='bonferroni')

pairwise_mt_within_group['p-Bonf'] = pvals_mt_bonf
pairwise_rt_within_group['p-Bonf'] = pvals_rt_bonf

cols = [
    'Contrast', 'phase', 'A', 'B', 'T', 'hedges', 'p-unc', 'p-FDR', 'p-Bonf'
]

print("Movement Time Pairwise Comparisons (FDR and Bonferroni):")
display(pairwise_mt_within_group[cols].round(4))

print("Response Time Pairwise Comparisons (FDR and Bonferroni):")
display(pairwise_rt_within_group[cols].round(4))

Movement Time Pairwise Comparisons (FDR and Bonferroni):


,Contrast,phase,A,B,T,hedges,p-unc,p-FDR,p-Bonf
0,phase,-,baseline_meanMT,early_learning_meanMT,-6.1645,-1.1852,0.0000,0.0000,0.0000
1,phase,-,baseline_meanMT,late_learning_meanMT,-4.0980,-0.8733,0.0004,0.0015,0.0031
2,phase,-,early_learning_meanMT,late_learning_meanMT,2.7491,0.4022,0.0114,0.0267,0.0800
3,group,-,adults,children,1.0299,0.4058,0.3145,0.4402,1.0000
4,phase * group,baseline_meanMT,adults,children,0.3477,0.1352,0.7313,0.8532,1.0000
5,phase * group,early_learning_meanMT,adults,children,2.3759,0.9616,0.0285,0.0498,0.1992
6,phase * group,late_learning_meanMT,adults,children,0.1773,0.0718,0.8612,0.8612,1.0000


Response Time Pairwise Comparisons (FDR and Bonferroni):


,Contrast,phase,A,B,T,hedges,p-unc,p-FDR,p-Bonf
0,phase,-,baseline_meanRT,early_learning_meanRT,1.3626,0.1073,0.1862,0.2607,1.0000
1,phase,-,baseline_meanRT,late_learning_meanRT,0.7728,0.0836,0.4475,0.5221,1.0000
2,phase,-,early_learning_meanRT,late_learning_meanRT,-0.2916,-0.0221,0.7732,0.7732,1.0000
3,group,-,adults,children,-2.8300,-1.1832,0.0136,0.0379,0.0950
4,phase * group,baseline_meanRT,adults,children,-2.7532,-1.1379,0.0145,0.0379,0.1017
5,phase * group,early_learning_meanRT,adults,children,-2.7712,-1.1684,0.0162,0.0379,0.1137
6,phase * group,late_learning_meanRT,adults,children,-2.5832,-1.0847,0.0225,0.0395,0.1578
